In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

# Load pretrained GPT-2 model and tokenizer
model = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token  # ensure pad token is defined

# Example training data: a list of dicts with "prompt" and "response"
train_data = [
    {"prompt": "User: How do I bake a cake?\nAssistant:", 
     "response": " Sure! To bake a cake, first preheat your oven to 350°F... [detailed steps]"},
    {"prompt": "User: What is the capital of France?\nAssistant:", 
     "response": " The capital of France is Paris."},
    # ... (more examples)
]

train_encodings = {"input_ids": [], "attention_mask": [], "labels": []}

# Convert to a format suitable for training (concatenate prompt and response)
max_len = 64
for ex in train_data:
    prompt = ex["prompt"]
    response = ex["response"]

    # Tokenize the training data
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response, add_special_tokens=False)["input_ids"]
    
    input_ids = prompt_ids + response_ids
    attention_mask = [1] * '''TO DO'''

    # Mask the prompt so the model learns the response
    labels = [-100] * '''TO DO''' + response_ids

    padding = max_len - len(input_ids)
    if max_len > len(input_ids):
        attention_mask += [0] * padding
        labels += '''TO DO''' * padding
        input_ids += [tokenizer.pad_token_id] * padding
    else:
        input_ids = input_ids[:max_len]
        attention_mask = attention_mask[:max_len]
        labels = labels[:max_len]

    train_encodings["input_ids"].append(input_ids)
    train_encodings["attention_mask"].append(attention_mask)
    train_encodings["labels"].append(labels)

/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [88]:
import torch
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        # Labels are the same as input_ids for causal LM
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.encodings["labels"][idx])
        }

train_dataset = TextDataset(train_encodings)
print(len(train_dataset))

2


In [ ]:
# Define training arguments (small training for demo purposes)
training_args = TrainingArguments(
    output_dir="sft-model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=10,
    learning_rate=5e-5,
    fp16=False,
    report_to=None  # no integrated logging
)

# Create a Trainer
trainer = Trainer(
    model='''TO DO''',
    args='''TO DO''',
    train_dataset='''TO DO'''  # in practice, prepare a proper Dataset object
)

# Fine-tune the model (this will run for a few epochs)
trainer.train()

  0%|          | 0/3 [53:30<?, ?it/s]
/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
  0%|          | 0/3 [00:00<?, ?it/s]/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  #           together with a few implementation details of multiprocessing, forces
100%|██████████| 3/3 [00:00<00:00,  8.11it/s]

{'train_runtime': 0.369, 'train_samples_per_second': 16.262, 'train_steps_per_second': 8.131, 'train_loss': 2.294468402862549, 'epoch': 3.0}


TrainOutput(global_step=3, training_loss=2.294468402862549, metrics={'train_runtime': 0.369, 'train_samples_per_second': 16.262, 'train_steps_per_second': 8.131, 'train_loss': 2.294468402862549, 'epoch': 3.0})

In [ ]:
from transformers import pipeline

# Use a sentiment analysis pipeline as a proxy reward model
sentiment_pipe = pipeline('sentiment-analysis')

def compute_reward(response_text):
    """Compute a reward score given the response text."""
    result = sentiment_pipe(response_text, truncation=True)[0]  # analyze sentiment
    if result['label'] == 'POSITIVE':
        # reward is +1 times confidence if positive
        return result['score']  
    else:
        # reward is -1 times confidence if negative
        return '''TO DO'''

No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


In [91]:
from trl import AutoModelForCausalLMWithValueHead, PPOTrainer, PPOConfig, create_reference_model

# Load the fine-tuned model into a value-head model for PPO
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained("sft-model")  # using the SFT model weights
ppo_model.to("mps")
tokenizer.pad_token = tokenizer.eos_token  # ensure tokenizer has a pad token set
# Create a reference model (for KL penalty) – usually a copy of initial policy
ref_model = create_reference_model(ppo_model)

# Set up PPO configuration
ppo_config = PPOConfig(
    batch_size=1,  # how many prompts per PPO update
    mini_batch_size=1,
    optimize_cuda_cache=False
)
ppo_trainer = PPOTrainer(config=ppo_config, model=ppo_model, ref_model=ref_model, tokenizer=tokenizer)

/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:257: UserWarning: No dataset is provided. Make sure to set config.batch_size to the correct value before training.
  warnings.warn(


In [ ]:
# Sample prompt for the model to respond to
prompt = "User: I don't think my job interview went well.\nAssistant:"

# Tokenize prompt and move to device
input_ids = [tokenizer(prompt, return_tensors="pt").input_ids[0].to("mps")]

# Generate a response from the current policy model
gen_output = ppo_trainer.generate(input_ids, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
response_ids = gen_output[0]
response_text = tokenizer.decode(response_ids, skip_special_tokens=True)
print("Model response before RLHF:", response_text)

# Compute reward using our proxy reward function
reward_score = '''TO DO'''
reward_tensor = [torch.tensor(reward_score, dtype=torch.float32, device="mps")]

# Run a PPO optimization step
train_stats = ppo_trainer.step(input_ids, [response_ids], reward_tensor)

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Model response before RLHF: User: I don't think my job interview went well.
Assistant: I'm sorry, but I'm not sure if you're sure.
Assistant: I'm sorry, but I'm not sure if you're sure.
Assistant: I'm sorry, but I'm not sure if you're sure.
Assistant


/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:1302: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  stats["tokens/queries_len_std"] = torch.std(query_lens).cpu().numpy().item()
/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:1305: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  stats["tokens/responses_len_std"] = torch.std(response_lens).cpu().numpy().item()


In [93]:
# Generate from model after PPO update
input_ids_tensor = torch.stack(input_ids).to("mps")
attention_mask_tensor = (input_ids_tensor != tokenizer.pad_token_id).long()
new_response_ids = ppo_model.generate(
    input_ids_tensor,
    attention_mask=attention_mask_tensor,
    max_new_tokens=50,
    pad_token_id=tokenizer.eos_token_id,
    top_k=50,
    top_p=0.95)
new_response_text = tokenizer.decode(new_response_ids[0], skip_special_tokens=True)
print("Model response after one PPO step:", new_response_text)

Model response after one PPO step: User: I don't think my job interview went well.
Assistant: I'm not sure if you're a good at the same level as me.
Assistant: I'm not sure if you're a good person.
Assistant: I'm not sure if you're a good person.
Assistant: I'm not


In [94]:
from more_itertools import chunked

batch_size = 2
prompts = ["User: How was your day?\nAssistant:",
           "User: I don't like my job.\nAssistant:",
           "User: Tell me a joke.\nAssistant:",
           "User: I need advice on studying.\nAssistant:",
           "User: What's the weather like?\nAssistant:"]

for epoch in range(5):
    for prompt in prompts:
        encoding = tokenizer(prompt, return_tensors='pt')

        query = encoding.input_ids[0].to("mps")

        attention_mask = encoding.attention_mask.to("mps")

        response = ppo_trainer.generate(
            query,
            attention_mask=attention_mask,
            max_new_tokens=50,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95)

        response = response.squeeze(0) if response.dim() == 2 else response
    
        text = tokenizer.decode(response, skip_special_tokens=True)
        reward = torch.tensor(compute_reward(text), dtype=torch.float32)

        train_states = ppo_trainer.step(
            [query.cpu()],
            [response.cpu()],
            [reward.cpu()]
        )

        print("Prompt:", prompt)
        print("Response:", text)
        print("Reward:", reward.item())
        print("PPO step done\n")

/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -13.48 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(


Prompt: User: How was your day?
Assistant:
Response: User: How was your day?
Assistant: Um, how was it?
Assistant: I suppose, uh, there was?
Assistant: Oh, absolutely! Uh, yes!
Assistant: Oh, that was wonderful. I've never been to the cafeteria before, uh, and um
Reward: 0.9995054006576538
PPO step done



/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -36.09 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(


Prompt: User: I don't like my job.
Assistant:
Response: User: I don't like my job.
Assistant: Hmm?
Assistant: Maybe you're on a mission here?
Assistant: Let's say you're back from your mission.
(Pause)
Assigns: Assigns: [Assistant]* <Assigns::_Person
Reward: -0.9983939528465271
PPO step done



/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -15.99 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(


...

/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -78.92 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(


Prompt: User: Tell me a joke.
Assistant:
Response: User: Tell me a joke.
Assistant: [Male] Yeah, I've heard that too. I don't know,
I guess that's right. Assistant: [Male] Hey, is it. How do you do?
Assistant: [Female] [Female] [Helping
Reward: -0.9856180548667908
PPO step done



/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -97.40 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(


Prompt: User: I need advice on studying.
Assistant:
Response: User: I need advice on studying.
Assistant: Right! How did you become a teacher?

Assistant: So, it was the teacher.

[A1F2P:Yes.]
[Q1F2P1:Yes!]

Assistant: You? Are you
Reward: -0.8809152245521545
PPO step done

Prompt: User: What's the weather like?
Assistant:
Response: User: What's the weather like?
Assistant::
The weather is good,:-)The weather'sGood(:-)So:-)You'll see:-)You'll see:-)So:-)So:-)

Also:-)You'll see:-)So:-)Then:-
Reward: 0.9980210065841675
PPO step done



/Users/sanchalibanerjee/Documents/AI@MIT Workshops/RLHF/venv/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -68.28 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(
